> ## SOLUTION / ANSWER KEY &mdash; Lab 6.9
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-09-multi-tool.ipynb`](../lab-09-multi-tool.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.9 &mdash; Multi-Tool Orchestration (day in the life)

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Give the agent TWO tools: a search and a calculator
- Run a real compound query and read the chained trace
- Check the compute step was grounded in the search result

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. Read the **Concept**, fill the real `___` blanks in **Build it** (real tool bodies, real prompts, the real `create_agent` call), then **Run it for real** &mdash; a real LLM drives a real agent over real tools &mdash; and **read the trace** to see exactly what it did. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`) and a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`). If Ollama is down, the run cells print how to start it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* aborts the whole agent run (you'll see exactly this in Lab 11).

**Reference:** [Module 6 slides &mdash; Connecting to real tools & APIs](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))   # SERPER_API_KEY, WOLFRAM_ALPHA_APPID, GROQ/OPENAI keys

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-06-09")
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model. Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Agents earn their keep on **multi-step** tasks that **orchestrate several tools** (deck slide 16):
*"population of France, divided by 2?"* needs a **search** (a live fact) **then** a **calculator**. You
bind both tools to one agent; the agent chains them &mdash; each observation feeds the next step. After
the run you'll **check the chain was grounded**: did the calculator actually operate on the number search
returned, or hallucinate one?

In [ ]:
from langchain_core.tools import tool

@tool
def web_search(query: str) -> str:
    """Search the web for a current fact or figure."""
    facts = {"population of france": "68000000"}
    return facts.get(query.lower().strip(), "no result")

print("a search tool:", web_search.name, "->", web_search.invoke("population of france"))

## Build it
Give the agent **both** tools in `build_agent`, and complete `chain_is_grounded` &mdash; the check that the
calculator step's argument was **derived from** the search result (real validation, not an LLM stand-in).

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import ToolMessage

@tool
def web_search(query: str) -> str:
    """Search the web for a current fact or figure. Use for population, prices, live facts."""
    facts = {"population of france": "68000000"}
    return facts.get(query.lower().strip(), "no result")

@tool
def calculator(expression: str) -> str:
    """Do exact arithmetic on an expression such as 68000000/2."""
    try:
        return str(safe_calc(expression))
    except Exception:
        return "error: not a numeric expression"

def build_agent():
    tools = [web_search, calculator]
    return create_agent(llm, tools)

def chained_arg(messages):
    """Pull (first search observation, calculator expression) out of the real trace."""
    obs = next((str(m.content) for m in messages if isinstance(m, ToolMessage)), None)
    expr = None
    for m in messages:
        for tc in (getattr(m, "tool_calls", None) or []):
            if tc["name"] == "calculator":
                expr = str(tc["args"].get("expression", ""))
    return obs, expr

def chain_is_grounded(messages):
    obs, expr = chained_arg(messages)
    return bool(obs and expr and obs in expr)

agent = build_agent()

try:
    print("agent nodes:", set(agent.nodes) - {"__start__"})
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Run the two-tool agent on a compound question, read the chained trace, and check whether it grounded the compute step in the search result.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        result = agent.invoke(
            {"messages": [("user", "Use web_search for the population of france, then use the calculator to halve it.")]},
            config={"recursion_limit": 8})
        print_trace(result)
        print("---")
        print("data dependency:", chained_arg(result["messages"]))
        print("grounded?      :", chain_is_grounded(result["messages"]))
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- A **grounded** run shows the calculator expression containing the exact number search returned (e.g. `68000000/2`).
- If `grounded?` is False, the model computed on a number it made up &mdash; a real, common failure you can now *detect* from the trace.
- Chaining search + compute in one run is exactly where an agent beats a single model call.

## Your turn (open task &mdash; no grader)
Ask a different compound question (e.g. *"population of france, times 3"*) and re-run. **What good looks
like:** the trace chains `web_search` -> `calculator`, and `chain_is_grounded` is True because the compute
used the searched figure. If it's False, read the trace to see where the model went off the rails.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
if ollama_up():
    result = agent.invoke(
        {"messages": [("user", "Use web_search for the population of france, then use the calculator to multiply it by 3.")]},
        config={"recursion_limit": 8})
    print_trace(result)
    print("data dependency:", chained_arg(result["messages"]))
    print("grounded?      :", chain_is_grounded(result["messages"]))
else:
    print("(start Ollama: ollama run llama3.1:8b)")

---
### Key takeaway
Chaining search + compute in one run is where agents beat a single call. Same create_agent, two tools -- plus a grounding check that catches a hallucinated chain.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>